In [5]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

def process_one_ihdp_rep(
    ihdp_path: str,
    rep_id: int,
    norm_dir: str = './ihdp_norm_data/',
    mask_dir: str = './ihdp_mask_data/',
):
    """
    Preprocess one IHDP rep file: normalize + create mask like ACIC 2016.
    Output filenames: ihdp_{rep_id}.csv

    Parameters:
    - ihdp_path: full path to ihdp_npci_{rep_id}.csv
    - rep_id: int ID of the rep file
    - norm_dir: output directory for normalized data
    - mask_dir: output directory for mask
    """
    out_name = f"ihdp_{rep_id}"
    df = pd.read_csv(ihdp_path, header=None, names=['treatment', 'y_factual', 'y_cfactual', 'mu0', 'mu1'] + [f'x{i}' for i in range(1, 26)])

    # Split columns
    covariate_cols = [f'x{i}' for i in range(1, 26)]
    outcome_cols = ['y_factual', 'y_cfactual', 'mu0', 'mu1']
    treatment_col = 'treatment'

    # Normalize covariates
    cov_scaler = StandardScaler()
    x_scaled = cov_scaler.fit_transform(df[covariate_cols])

    # Normalize outcomes
    y_scaler = StandardScaler()
    y_all = np.vstack([df['y_factual'], df['y_cfactual']]).reshape(-1, 1)
    y_scaler.fit(y_all)
    y_factual_scaled = y_scaler.transform(df['y_factual'].values.reshape(-1, 1))
    y_cfactual_scaled = y_scaler.transform(df['y_cfactual'].values.reshape(-1, 1))
    mu0_scaled = y_scaler.transform(df['mu0'].values.reshape(-1, 1))
    mu1_scaled = y_scaler.transform(df['mu1'].values.reshape(-1, 1))

    z = df['treatment'].values.reshape(-1, 1)

    full_data = np.hstack([z, y_factual_scaled, y_cfactual_scaled, mu0_scaled, mu1_scaled, x_scaled])

    # Create mask: observed = 1, unobserved = 0
    mask = np.ones_like(full_data)
    print(mask.shape)
    mask[:, 3] = 0  # mu0
    mask[:, 4] = 0  # mu1
    for i in range(full_data.shape[0]):
        if full_data[i, 0] == 0:  # control group, y_factual = y0, mask y_cfactual
            mask[i, 2] = 0
        else:  # treated, y_factual = y1, mask y_factual
            mask[i, 1] = 0

    # Save
    os.makedirs(norm_dir, exist_ok=True)
    os.makedirs(mask_dir, exist_ok=True)
    

    pd.DataFrame(full_data).to_csv(os.path.join(norm_dir, f'{out_name}.csv'), index=False, header=False)
    pd.DataFrame(mask).to_csv(os.path.join(mask_dir, f'{out_name}.csv'), index=False, header=False)

    print(f"[✓] Saved: {out_name}.csv")

In [6]:
def batch_process_ihdp_all_reps(
    base_dir='ihdp_data',
    norm_dir='./ihdp_norm_data/',
    mask_dir='./ihdp_mask_data/',
):
    for rep_id in range(1, 11):  # ihdp_npci_1 to 10
        filename = f'ihdp_npci_{rep_id}.csv'
        ihdp_path = os.path.join(base_dir, filename)
        if not os.path.exists(ihdp_path):
            print(f"[!] File not found: {ihdp_path}")
            continue

        process_one_ihdp_rep(
            ihdp_path=ihdp_path,
            rep_id=rep_id,
            norm_dir=norm_dir,
            mask_dir=mask_dir
        )

In [7]:
# Run the preprocessing
batch_process_ihdp_all_reps(
    base_dir='ihdp_data',  # path to the folder with your IHDP CSVs
    norm_dir='./ihdp_norm_data/',
    mask_dir='./ihdp_mask_data/',
)

(747, 30)
[✓] Saved: ihdp_1.csv
(747, 30)
[✓] Saved: ihdp_2.csv
(747, 30)
[✓] Saved: ihdp_3.csv
(747, 30)
[✓] Saved: ihdp_4.csv
(747, 30)
[✓] Saved: ihdp_5.csv
(747, 30)
[✓] Saved: ihdp_6.csv
(747, 30)
[✓] Saved: ihdp_7.csv
(747, 30)
[✓] Saved: ihdp_8.csv
(747, 30)
[✓] Saved: ihdp_9.csv
(747, 30)
[✓] Saved: ihdp_10.csv
